# The SVD in machine learning
PCA through the SVD, latent semantic analysis, a small ratings matrix, and least squares with the pseudo-inverse.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LinearRegression
np.set_printoptions(precision=3, suppress=True)

## 1. PCA through the SVD
The made-up flats data (rooms, washrooms) of the PCA Notes, rebuilt with the same random seed.

In [ ]:
rng = np.random.default_rng(7)
N = 30
rooms = rng.uniform(1, 5, N)
rng.normal(0, 0.35, N)        # keeps the random stream identical to the PCA Notes
wash = 0.8 * rooms + rng.normal(0, 0.35, N) + 0.4
wash = wash * rooms.std() / wash.std()
wash += 3 - wash.mean()
P = np.c_[rooms, wash]
Xc = P - P.mean(axis=0)       # step 1: centre each column

In [ ]:
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
print('singular values:', s)
print('sigma^2 / n (variances, as in the PCA Notes):', s ** 2 / N)
print('principal components (rows of V^T):\n', Vt)
print('scores of the first 3 flats (U Sigma):\n', (U * s)[:3])

In [ ]:
# the same through the covariance matrix (divide by n, as the PCA Notes do)
print(np.linalg.eigvalsh(np.cov(P.T, bias=True))[::-1])
# scikit-learn divides by n - 1
pca = PCA(n_components=2).fit(P)
print(pca.explained_variance_, s ** 2 / (N - 1))
print('solver chosen by svd_solver="auto":', pca._fit_svd_solver)
print('scores equal up to sign:', np.allclose(np.abs(pca.transform(P)), np.abs(U * s)))

## 2. Latent semantic analysis

In [ ]:
docs = ['batsman hits a century', 'bowler takes a wicket', 'batsman and bowler: century, wicket, match',
        'chef cooks a curry', 'rice and dal', 'chef serves curry with rice and dal', 'chef watches the match']
cv = CountVectorizer(stop_words='english')
X = cv.fit_transform(docs)
print(cv.get_feature_names_out())
print('word-count cosine similarity:\n', cosine_similarity(X).round(2))

In [ ]:
lsa = TruncatedSVD(n_components=2, random_state=0)
Z = lsa.fit_transform(X)       # U_k Sigma_k: each document in 2 topic coordinates
print('singular values:', lsa.singular_values_)
print(Z.round(3))
print('topic-space cosine similarity:\n', cosine_similarity(Z).round(2))

## 3. A ratings matrix

In [ ]:
R = np.array([[5, 4, 1, 1], [4, 5, 2, 1], [1, 1, 5, 4], [2, 1, 4, 5], [5, 5, 4, 4]], float)
U, s, Vt = np.linalg.svd(R, full_matrices=False)
print('singular values:', s)
print('v1 (overall liking):', Vt[0])
print('v2 (action vs romance):', Vt[1])
R2 = (U[:, :2] * s[:2]) @ Vt[:2]
print(R2.round(2))
print('largest difference from R:', np.abs(R - R2).max().round(2))

## 4. Least squares with the pseudo-inverse
The four students of the multiple linear regression maths Note: CGPA and package.

In [ ]:
cgpa = np.array([6.89, 5.12, 7.82, 7.42])
y = np.array([3.26, 1.98, 3.25, 3.67])
X = np.c_[np.ones(4), cgpa]
U, s, Vt = np.linalg.svd(X, full_matrices=False)
X_plus = Vt.T @ np.diag(1 / s) @ U.T      # V Sigma^-1 U^T
print('beta via pseudo-inverse:', X_plus @ y)
print('beta via normal equation:', np.linalg.inv(X.T @ X) @ X.T @ y)
print('same as np.linalg.pinv:', np.allclose(X_plus, np.linalg.pinv(X)))

In [ ]:
# add a column that is exactly 9.5 x CGPA (a percentage)
X3 = np.c_[np.ones(4), cgpa, 9.5 * cgpa]
print('singular values:', np.linalg.svd(X3, compute_uv=False))
b_inv = np.linalg.inv(X3.T @ X3) @ X3.T @ y
print('inverse (no error raised!):', b_inv, ' predictions:', X3 @ b_inv)
b_pinv = np.linalg.pinv(X3) @ y
print('pseudo-inverse:', b_pinv, ' predictions:', X3 @ b_pinv)
print('lstsq:', np.linalg.lstsq(X3, y, rcond=None)[0])
lr = LinearRegression().fit(X3[:, 1:], y)
print('LinearRegression:', lr.intercept_, lr.coef_)